In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from encoder_decoder import*

# Playing around with encodings and embeddings to learn how to use them

In [18]:
# first import the little dictionary frol it_es_cognates.txt

italian_words = []
spanish_words = []

with open("it_es_cognates.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        it, es = line.split(";") # words separated by ;
        italian_words.append(it.lower())
        spanish_words.append(es.lower())

# check the number of words in the list of cognates
print(len(italian_words), "pairs")
print(italian_words[:5], spanish_words[:5])

# find all characters appearing in the list of cognates
all_chars = set()
for word in spanish_words + italian_words:
    all_chars.update(word)
all_chars = sorted(all_chars)

# add the special caracters: pad, start of string, end of string, unknown
specials = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab = specials + all_chars

# build the dictionaries, just use the enumeration of vocab to assign an integer to every character
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# check the characters in the vocabulary, and its length
print("vocab: ", vocab)
print("vocab length:", len(vocab))

# test word size in the vocabulary (need to know the dimension of the words I am going to have in the model - by padding)
# I will add 2 or 3 just to be on the safe side for future additions to the dictionary, getting, say, to 20
print("Max length of spanish words: ", max(len(w) for w in spanish_words))
print("Max length of italian words: ", max(len(w) for w in italian_words))

def encode_source(word, max_len = 20):
    # input: no sos/eos needed
    ids = [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len] # use the char to int dictionary
    ids += [char_to_idx['<pad>']] * (max_len - len(ids)) # fill with '<pad>' until the prescribed length max_len
    return torch.tensor(ids)

def encode_target(word, max_len = 20):
    # output: needs sos/eos since decoder generates it step by step, they will replace two <pad> 
    ids = [char_to_idx['<sos>']] + [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len-2] + [char_to_idx['<eos>']]
    ids += [char_to_idx['<pad>']] * (max_len - len(ids))
    return torch.tensor(ids) 

def decode_source(ids):
    
    ids_numpy = ids.numpy()
    chars = []
    for i in ids_numpy:
        ch = idx_to_char[i] # use the integer to char dictionary
        if ch == '<pad>':
            break  # padding marks the end of real content
        chars.append(ch)
    return ''.join(chars)

# test
print(encode_source('castoro'))
print(decode_source(encode_source('castoro'))) 
print(decode_source(encode_source('è'))) # check unknown characters


# now test the embedding
vocab_size = len(char_to_idx)
d_model = 32 # needs to be large enough but does not have to be larger than number of characters - in fact for LLMs it is smaller than the number of tokens
embed = nn.Embedding(vocab_size, d_model, padding_idx=char_to_idx['<pad>'])

print(embed(encode_source('castoro')))

908 pairs
['acqua', 'aglio', 'aiutare', 'aiutata', 'aiutate'] ['agua', 'ajo', 'ayudar', 'ayudada', 'ayudadas']
vocab:  ['<pad>', '<sos>', '<eos>', '<unk>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', 'à', 'á', 'é', 'í', 'ñ', 'ó', 'ù', 'ú']
vocab length: 36
Max length of spanish words:  15
Max length of italian words:  13
tensor([ 6,  4, 21, 22, 17, 20, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0])
castoro
<unk>
tensor([[ 0.4125, -0.0494,  0.3327,  0.0076, -1.1486, -0.5626, -1.3342,  1.2032,
         -0.6590, -0.9502,  1.2344, -1.7797,  1.0961, -0.6409, -0.3750, -0.0291,
         -0.3938, -1.5672, -0.2439,  1.6152,  0.6817, -0.0993, -1.1109, -1.1292,
          0.5682, -0.7447,  0.6996, -1.2912,  1.7252, -2.6859,  0.5463, -0.3589],
        [-1.0292,  1.2285,  2.4279, -0.6764,  0.8403,  0.2182,  0.8308,  0.0322,
          0.5763, -0.3184,  0.1744, -0.7746,  0.5734, -0.1487,  0.6715, -0.2908,
   

In [ ]:
# test if Encoder runs correctly

enco = Encoder(2, d_model = d_model, d_hidden = 4*d_model, dk = 8, dv = 8, h = 4)

encoded_word = enco(embed(encode_source('castoro')).unsqueeze(0))
print(encoded_word)

#ok

tensor([[[ 0.5759,  0.0903,  0.4997,  0.0477, -1.1290,  0.1545, -1.3894,
           1.2152, -0.7081, -0.9169,  1.2823, -0.6205,  1.2538, -0.5593,
          -0.1206,  0.1753, -0.5490, -1.2402,  0.0042,  2.5334,  0.7491,
          -0.1499, -0.3177, -1.2346,  0.8201, -0.9878,  0.6477, -1.2899,
           2.0997, -1.4992,  0.7221, -0.1590],
         [-1.0623,  0.8815,  2.4944, -0.5498,  0.1243,  0.5096,  0.5796,
          -0.5941,  0.3457, -0.2028, -0.1580, -1.0030,  0.3597, -0.9032,
           0.7360, -1.0214,  1.6764,  0.6268,  0.5227,  2.4225,  0.0161,
          -1.2083, -1.1759, -0.6698, -0.5020, -0.2124, -0.5317, -0.2490,
          -0.5891,  1.5504, -0.6552, -1.5576],
         [ 1.5382, -1.8737,  0.9072,  0.8990, -1.4046,  0.4575,  0.2471,
          -0.9708,  1.6302, -0.3237,  0.8101, -0.4814, -0.5727,  0.8246,
          -0.8202, -0.4302, -0.1673, -0.3124,  0.0461,  0.0450,  0.2661,
          -0.7162,  2.0915, -2.2129, -0.5977, -1.2999, -0.6257, -0.0646,
           0.7976,  1.5320,  0